In [1]:
# ============================================================
# CELL 1 — Mount Google Drive
# ============================================================

# "drive" is a library that connects Google Colab to your
# Google Drive account. Think of it like a USB cable between
# Colab and your Drive.
from google.colab import drive

# This line actually connects them. When you run this, Google
# will ask you to click a link and give permission.
# After that, your Drive will appear at /content/drive/
drive.mount('/content/drive')

# Let's print a message so we know it worked
print("✅ Google Drive mounted successfully!")
print("Your Drive is now visible at: /content/drive/MyDrive/")

Mounted at /content/drive
✅ Google Drive mounted successfully!
Your Drive is now visible at: /content/drive/MyDrive/


In [2]:
# ============================================================
# CELL 2 — Install libraries we need
# ============================================================

# "pip install" means "download and install this tool"
# Think of it like installing an app on your phone

# opencv-python = a library for reading/writing videos and images
# We need this for our video/webcam inference in Step 8
!pip install opencv-python --quiet

# scikit-learn = a library for evaluation metrics like F1-score
# We need this for Step 6 (confusion matrix, classification report)
!pip install scikit-learn --quiet

# seaborn = a library for making beautiful plots/charts
# We need this for Step 6 (plotting the confusion matrix nicely)
!pip install seaborn --quiet

# These libraries come pre-installed in Colab, but let's
# make sure they are up to date
!pip install tensorflow --quiet
!pip install matplotlib --quiet

print("✅ All libraries installed successfully!")

✅ All libraries installed successfully!


In [3]:
# ============================================================
# CELL 3 — Locate and extract the dataset from Drive
# ============================================================

import os       # os = operating system tools (for file/folder work)
import zipfile  # zipfile = for opening .zip files

# ----------------------------------------------------------
# STEP A: Define the path to your zip file in Google Drive
# ----------------------------------------------------------
# This is the path you told me: Driver_Behaviour_Monitoring folder
# Change this if your folder name is different!

ZIP_PATH = "/content/drive/MyDrive/Driver_Behaviour_Monitoring/state-farm-distracted-driver-detection.zip"

# Where do we want to extract the dataset?
# /content/ is Colab's local fast storage (much faster than Drive)
EXTRACT_PATH = "/content/dataset/"

# ----------------------------------------------------------
# STEP B: Check if the zip file actually exists
# ----------------------------------------------------------
# os.path.exists() returns True if the file is there, False if not
if os.path.exists(ZIP_PATH):
    print(f"✅ Found zip file at: {ZIP_PATH}")
else:
    print(f"❌ ERROR: Zip file NOT found at: {ZIP_PATH}")
    print("Please check your Drive folder name and zip file name!")
    print("Make sure it matches exactly (capital letters matter!)")

✅ Found zip file at: /content/drive/MyDrive/Driver_Behaviour_Monitoring/state-farm-distracted-driver-detection.zip


In [4]:
# ============================================================
# CELL 4 — Extract the dataset (only if not already extracted)
# ============================================================

# We check first — if already extracted, we skip to save time
# This is important because extraction takes ~5 minutes!
# If Colab disconnects and you reconnect, it won't waste time
# extracting again if files are already there.

if not os.path.exists(EXTRACT_PATH):
    print("📦 Extracting dataset... this may take 3-5 minutes...")
    print("Please wait — do NOT close the browser!")

    # Open the zip file and extract everything inside it
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)

    print("✅ Dataset extracted successfully!")

else:
    # Already extracted — skip!
    print("✅ Dataset already extracted — skipping extraction.")
    print(f"   Found at: {EXTRACT_PATH}")

📦 Extracting dataset... this may take 3-5 minutes...
Please wait — do NOT close the browser!
✅ Dataset extracted successfully!


In [5]:
# ============================================================
# CELL 5 — Check what's inside the extracted dataset
# ============================================================

# Let's look at what folders and files we have
# os.listdir() = list everything inside a folder
# Like opening a folder in File Explorer

print("📂 Contents of dataset folder:")
print("=" * 40)

for item in sorted(os.listdir(EXTRACT_PATH)):
    item_path = os.path.join(EXTRACT_PATH, item)

    # os.path.isdir() = check if it's a folder (not a file)
    if os.path.isdir(item_path):
        print(f"  📁 {item}/")
    else:
        print(f"  📄 {item}")

print("=" * 40)

# Now let's look inside the 'imgs' folder which has train/test
IMGS_PATH = os.path.join(EXTRACT_PATH, "imgs")

if os.path.exists(IMGS_PATH):
    print("\n📂 Contents of imgs/ folder:")
    for item in sorted(os.listdir(IMGS_PATH)):
        print(f"  📁 {item}/")

    # Let's look inside 'train' folder and count classes
    TRAIN_PATH = os.path.join(IMGS_PATH, "train")
    if os.path.exists(TRAIN_PATH):
        classes = sorted(os.listdir(TRAIN_PATH))
        print(f"\n✅ Found {len(classes)} classes in train/:")
        for cls in classes:
            cls_path = os.path.join(TRAIN_PATH, cls)
            count = len(os.listdir(cls_path))
            print(f"   {cls} → {count} images")

📂 Contents of dataset folder:
  📄 driver_imgs_list.csv
  📁 imgs/
  📄 sample_submission.csv

📂 Contents of imgs/ folder:
  📁 test/
  📁 train/

✅ Found 10 classes in train/:
   c0 → 2489 images
   c1 → 2267 images
   c2 → 2317 images
   c3 → 2346 images
   c4 → 2326 images
   c5 → 2312 images
   c6 → 2325 images
   c7 → 2002 images
   c8 → 1911 images
   c9 → 2129 images


In [6]:
# ============================================================
# CELL 6 — Define all paths we will use throughout the project
# ============================================================
# Think of this as a "settings file" — we define everything
# here once, so we never have to search for paths later.

# ---- Dataset paths ----------------------------------------
DATASET_PATH  = "/content/dataset/"
TRAIN_PATH    = "/content/dataset/imgs/train/"
TEST_PATH     = "/content/dataset/imgs/test/"

# ---- Where to save our model checkpoints on Drive ----------
# We save to Drive so checkpoints survive if Colab restarts!
CHECKPOINT_DIR  = "/content/drive/MyDrive/Driver_Behaviour_Monitoring/checkpoints/"
CHECKPOINT_PATH = CHECKPOINT_DIR + "mobilenetv2_best.keras"

# ---- Model settings (we'll use these in Steps 4 & 5) -------
IMAGE_SIZE  = (224, 224)   # MobileNetV2 expects 224x224 pixels
BATCH_SIZE  = 16           # 16 images at a time (low = less memory)
EPOCHS      = 20           # Train for max 20 rounds
NUM_CLASSES = 10           # 10 distraction categories
LEARNING_RATE = 0.001      # How fast the model learns

# ---- Class names (what each folder c0-c9 means) ------------
CLASS_NAMES = {
    'c0': 'Safe driving',
    'c1': 'Texting - right',
    'c2': 'Phone call - right',
    'c3': 'Texting - left',
    'c4': 'Phone call - left',
    'c5': 'Operating radio',
    'c6': 'Drinking',
    'c7': 'Reaching behind',
    'c8': 'Hair or makeup',
    'c9': 'Talking to passenger'
}

# ---- Create checkpoint folder if it doesn't exist ----------
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ---- Print summary -----------------------------------------
print("✅ All paths and settings configured!")
print(f"\n📁 Train path     : {TRAIN_PATH}")
print(f"📁 Test path      : {TEST_PATH}")
print(f"💾 Checkpoint path: {CHECKPOINT_PATH}")
print(f"\n⚙️  Image size  : {IMAGE_SIZE}")
print(f"⚙️  Batch size  : {BATCH_SIZE}")
print(f"⚙️  Epochs      : {EPOCHS}")
print(f"⚙️  Num classes : {NUM_CLASSES}")

✅ All paths and settings configured!

📁 Train path     : /content/dataset/imgs/train/
📁 Test path      : /content/dataset/imgs/test/
💾 Checkpoint path: /content/drive/MyDrive/Driver_Behaviour_Monitoring/checkpoints/mobilenetv2_best.keras

⚙️  Image size  : (224, 224)
⚙️  Batch size  : 16
⚙️  Epochs      : 20
⚙️  Num classes : 10
